In [2]:
import pandas as pd

pd.set_option("display.max_columns", 100)

maps = pd.read_csv("C:\\Users\\Nico\\Documents\\The Power MBA\\Proyecto Final\\cs2-pro-scene-analysis\\data\\processed\\games_maps_clean.csv", parse_dates=["datetime"])
print(maps.shape)

(12444, 103)


In [3]:
stats = ["kills", "deaths", "assists", "adr", "kast", "kddiff"]
partes = []

for lado in (1, 2):
    rival = 2 if lado == 1 else 1
    for slot in range(1, 6):
        pre = f"team{lado}_player{slot}"
        d = pd.DataFrame({
            "match_id": maps["match_id"], "game_id": maps["game_id"],
            "tournament": maps["tournament"], "tier": maps["tier"],
            "map_name": maps["map_name"], "datetime": maps["datetime"],
            "year": maps["year"], "month": maps["month"], "best_of": maps["best_of"],
            "team_id": maps[f"team{lado}_id"], "team": maps[f"team{lado}"],
            "opponent_id": maps[f"team{rival}_id"], "opponent": maps[f"team{rival}"],
            "side": f"team{lado}",
            "rounds_won": maps[f"score{lado}_game"],
            "rounds_lost": maps[f"score{rival}_game"],
            "player_id": maps[f"{pre}_id"], "player_raw": maps[pre],
        })
        for s in stats:
            d[s] = maps[f"{pre}_{s}"]
        partes.append(d)

long = pd.concat(partes, ignore_index=True)
print(long.shape)

(124440, 24)


In [4]:
print(long[["player_id", "player_raw", "kills"]].isna().sum())
long = long.dropna(subset=["player_id", "player_raw", "kills"]).copy()
print(long.shape)

player_id     3847
player_raw    5316
kills         5316
dtype: int64
(119124, 24)


In [5]:
print("duplicados jugador-mapa:", long.duplicated(["game_id", "player_id"]).sum())
long = long.drop_duplicates(subset=["game_id", "player_id"], keep="first")
print(long.shape)

duplicados jugador-mapa: 19
(119105, 24)


In [6]:
patron = r"^(?P<nick>.+?)(?:\s\((?P<full_name>.*?)\))?(?:,\s*(?P<country>[A-Za-z]{2}))?$"
extra = long["player_raw"].str.extract(patron)
long = pd.concat([long, extra], axis=1).drop(columns="player_raw")
print(long[["nick", "full_name", "country"]].isna().sum())

nick             0
full_name    12140
country       9153
dtype: int64


In [7]:
long["map_win"] = (long["rounds_won"] > long["rounds_lost"]).astype(int)
long["total_rounds"] = long["rounds_won"] + long["rounds_lost"]

for col in ["game_id", "player_id", "rounds_won", "rounds_lost", "total_rounds"]:
    long[col] = long[col].astype("int64")
long["best_of"] = long["best_of"].astype("Int64")

print(long.shape)
print("kddiff = kills - deaths:", (long["kddiff"] == long["kills"] - long["deaths"]).mean())
print("jugadores (ids):", long["player_id"].nunique(), "| nicks:", long["nick"].nunique())
print(long["country"].value_counts().head(5))

(119105, 28)
kddiff = kills - deaths: 1.0
jugadores (ids): 1339 | nicks: 1337
country
RU    17167
DK     9977
BR     9000
UA     7194
SE     6350
Name: count, dtype: int64


In [8]:
long.to_csv("C:/Users/Nico/Documents/The Power MBA/Proyecto Final/cs2-pro-scene-analysis/data/processed/players_maps_long.csv", index=False)